In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
prev_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\YTD 2024-2025 NC_E2.xlsx", 
                        parse_dates=['Invoice Date'])
curr_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\SAP_apr1st_may31st_NCE2.xlsx",
                        parse_dates=['Invoice Date'])

In [3]:
prev_df.columns = prev_df.columns.str.strip()
curr_df.columns = curr_df.columns.str.strip()

In [4]:
prev_df.columns

Index(['Channel', 'MASTER', 'AREA', 'CATEGORY', 'VARIANTS', 'Invoice Type',
       'Invoice Date', 'Customer Code', 'Customer Name', 'Item Category',
       'Item Code', 'Item Name', 'Qty', 'UOM', 'Gross Weight',
       'Billing Amount', 'C. No', 'C. Name', 'C. Area'],
      dtype='object')

In [5]:
curr_df.columns

Index(['Invoice Date', 'Distribution Channel', 'Material Group',
       'Material Sub Group', 'Item Code', 'Item Name', 'Invoice Qty', 'UOM',
       'Gross Weight', 'Bill Amount', 'C. No', 'C. Name', 'C. Area'],
      dtype='object')

In [6]:
prev_df['C. No'] = prev_df['C. No'].astype(str).str.strip()
curr_df['C. No'] = curr_df['C. No'].astype(str).str.strip()

In [7]:
prev_df['Item Code'] = prev_df['Item Code'].astype(str).str.strip()
curr_df['Item Code'] = curr_df['Item Code'].astype(str).str.strip()

In [8]:
# Define Time Windows
analysis_month = pd.to_datetime("2025-05-01")
this_month = analysis_month.to_period("M").strftime('%Y-%m')
last_month = (analysis_month - pd.DateOffset(months=1)).to_period("M").strftime('%Y-%m')
last_year_same_month = (analysis_month - pd.DateOffset(years=1)).to_period("M").strftime('%Y-%m')

In [9]:
prev_df['period'] = prev_df['Invoice Date'].dt.to_period("M").astype(str)
curr_df['period'] = curr_df['Invoice Date'].dt.to_period("M").astype(str)

In [10]:
# Aggregate Sales by Distributor and Item
# Previous year same month
last_year_sales = prev_df[prev_df['period'] == last_year_same_month]
last_year_grouped = last_year_sales.groupby(['C. No', 'Item Code'])['Qty'].sum().reset_index()
last_year_grouped.rename(columns={'Qty': 'last_year_qty'}, inplace=True)

In [11]:
# Current year April & May (2 months)
curr_2_months = curr_df[curr_df['period'].isin([last_month, this_month])]
curr_grouped = curr_2_months.groupby(['C. No', 'Item Code'])['Invoice Qty'].sum().reset_index()
curr_grouped.rename(columns={'Invoice Qty': 'recent_2mo_qty'}, inplace=True)

In [12]:
# Merge for comparison
reco_df = pd.merge(last_year_grouped, curr_grouped, on=['C. No', 'Item Code'], how='left')
reco_df['recent_2mo_qty'] = reco_df['recent_2mo_qty'].fillna(0)

In [13]:
# Add Item Name and Distributor Name
item_lookup = pd.concat([
    prev_df[['Item Code', 'Item Name']],
    curr_df[['Item Code', 'Item Name']]
]).drop_duplicates()
reco_df = reco_df.merge(item_lookup, on='Item Code', how='left')

In [14]:
dist_lookup = pd.concat([
    prev_df[['C. No', 'C. Name']],
    curr_df[['C. No', 'C. Name']]
]).drop_duplicates()
reco_df = reco_df.merge(dist_lookup, on='C. No', how='left')

In [15]:
# Recommendation Condition: seasonal drop
reco_df['recommend'] = reco_df['last_year_qty'] > (1.5 * reco_df['recent_2mo_qty'])

In [16]:
# Final Result
recommended = reco_df[reco_df['recommend'] == True].copy()
recommended = recommended[['C. No', 'C. Name', 'Item Code', 'Item Name', 'last_year_qty', 'recent_2mo_qty']]
recommended = recommended.sort_values(by=['C. No', 'last_year_qty'], ascending=[True, False])

In [17]:
recommended.head(20)

,C. No,C. Name,Item Code,Item Name,last_year_qty,recent_2mo_qty
0,100006,MAHALAXMI BAKERY,1400919.0,15 KG 5-6 MM PAPAYA FRUIT PRESERVED RED POU (K...,160.0,0.0
1,100006,MAHALAXMI BAKERY,1400919.0,15Kg*1 5-6MM CANDIED FRUIT RED POU KRCHI,160.0,0.0
2,100007,MODEL BAKERS AND CONFECTIONERS,1400845.0,15 KG MODEL PAPAYA FRUIT PRESERVE MIX POU,300.0,0.0
3,100007,MODEL BAKERS AND CONFECTIONERS,1400845.0,15Kg*1 MODEL CANDIED FRUIT MIX POU,300.0,0.0
4,100013,RAM BAKERY,1400919.0,15 KG 5-6 MM PAPAYA FRUIT PRESERVED RED POU (K...,300.0,0.0
5,100013,RAM BAKERY,1400919.0,15Kg*1 5-6MM CANDIED FRUIT RED POU KRCHI,300.0,0.0
24,100015,S.M. SALES,1400840.0,15 KG POPULAR PAPAYA FRUIT PRESERVED MIX POU,32.0,0.0
25,100015,S.M. SALES,1400840.0,15Kg*1 POP CANDIED FRUIT MIX POU,32.0,0.0
22,100015,S.M. SALES,1400838.0,15 KG POPULAR PAPAYA FRUIT PRESERVED RED POU,30.0,0.0
23,100015,S.M. SALES,1400838.0,15Kg*1 POP CANDIED FRUIT RED POU,30.0,0.0


In [18]:
recommended['recent_2mo_qty'].value_counts()

recent_2mo_qty
0.0    25201
Name: count, dtype: int64